# Database Descriptive Stats

This notebook inspects one classification database/version and reports:
- Total number of papers
- Number of micro, meso, macro clusters
- For each cluster level, how many clusters (and %) have size greater than each threshold

In [ ]:
import awswrangler as wr
import pandas as pd

# Inputs
database = "cities"
version = "version3"

thresholds = [10000, 5000, 1000, 500, 300, 100, 50, 30, 10]

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

In [ ]:
def candidate_roots(database: str, version: str) -> list[str]:
    # Try versioned and non-versioned layouts to support both storage patterns.
    roots = []
    version = (version or "").strip()
    if version:
        roots.append(f"s3://openalex-outputs/classification/{database}/{version}/")
        roots.append(f"s3://openalex-outputs/classification/{database}_{version}/")
    roots.append(f"s3://openalex-outputs/classification/{database}/")
    return roots


def load_reports(database: str, version: str):
    tried = {}
    for root in candidate_roots(database, version):
        try:
            micro_df = wr.s3.read_parquet(f"{root}cluster_report_micro/")
            meso_df = wr.s3.read_parquet(f"{root}cluster_report_meso/")
            macro_df = wr.s3.read_parquet(f"{root}cluster_report_macro/")

            for name, df in [("micro", micro_df), ("meso", meso_df), ("macro", macro_df)]:
                if "publications" not in df.columns:
                    raise ValueError(f"Missing 'publications' column in {name} report at {root}")

            return root, micro_df, meso_df, macro_df
        except Exception as exc:
            tried[root] = str(exc)

    details = "\n".join([f"- {r}: {e}" for r, e in tried.items()])
    raise FileNotFoundError(
        "Could not load cluster reports from any candidate root.\n"
        f"Tried roots:\n{details}"
    )


source_root, micro_df, meso_df, macro_df = load_reports(database, version)
print(f"Loaded reports from: {source_root}")
print(f"Rows -> micro: {len(micro_df):,}, meso: {len(meso_df):,}, macro: {len(macro_df):,}")

In [ ]:
n_papers = int(pd.to_numeric(micro_df["publications"], errors="coerce").fillna(0).sum())

overview = pd.DataFrame(
    {
        "metric": ["papers", "micro_clusters", "meso_clusters", "macro_clusters"],
        "value": [n_papers, len(micro_df), len(meso_df), len(macro_df)],
    }
)

overview

In [ ]:
def threshold_stats(df: pd.DataFrame, level_name: str, thresholds: list[int]) -> pd.DataFrame:
    sizes = pd.to_numeric(df["publications"], errors="coerce").fillna(0)
    total_clusters = len(df)

    rows = []
    for t in thresholds:
        clusters_above = int((sizes > t).sum())
        pct_above = (100.0 * clusters_above / total_clusters) if total_clusters else 0.0
        rows.append(
            {
                "level": level_name,
                "threshold": t,
                "clusters_above_threshold": clusters_above,
                "pct_clusters_above_threshold": round(pct_above, 2),
                "total_clusters": total_clusters,
            }
        )

    return pd.DataFrame(rows)


threshold_summary = pd.concat(
    [
        threshold_stats(micro_df, "micro", thresholds),
        threshold_stats(meso_df, "meso", thresholds),
        threshold_stats(macro_df, "macro", thresholds),
    ],
    ignore_index=True,
)

threshold_summary